# Lab 2 — Predicting a Number, and Measuring It Honestly

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/INCORTX/INCORTX.github.io/blob/master/DataAnalytics/session-02/lab_02.ipynb)

**Data Analytics · Lab session 2**

Session 1 answered with a **label** — survived or not. This session the answer is a **number**, and
that changes how you measure everything: being wrong by 2 and being wrong by 200 are not the same mistake.

**The demo is one story in two acts.** Block A fits the least-squares line from Lecture 4 (Regression Analysis) to 20,640 California
districts and scores it — R² `0.5758`, a respectable number. Block B looks at what that number hid:
**a bend** the line cannot follow, **965 districts** whose true value was erased before we got the data,
and **one district of 36 people** that the line priced at 11.5 — \$1.15 million — when its true value was 1.6. The tool for all of block B is the residual plot — the one Lecture 4 spent its middle third on.

### How the session runs

| | Part | What happens |
|---|---|---|
| **1** | 🎤 **Assignment 1 on screen — 60 min** | **The hour opens with presentations of last session's work.** Two speakers per group, 5+ minutes, then two of questions. |
| **2** | 🎬 **Demo — 45 min** | Blocks A and B. The instructor walks it; you watch. Do not type along — you keep this file. |
| **3** | 📋 **Pick a topic** | Your group claims one of the fifteen. First come, first served. |
| **4** | 🟠 **Your hour — 60 min** | The section at the bottom. The same five steps as session 1, on a number. |

---
## 🎤 The 45 minutes we actually walk through

**This notebook's demo half holds about 78 minutes of material and the slot is 45.** The eight below are the ones we walk together — **about 45 minutes: the demo is full.** Everything else is reference you keep. *(This table is a map, not something read aloud.)*

| | Walked in the demo | Why this one earns the time |
|:--:|---|---|
| 1 | **How the session runs** | so the hour is not spent guessing what to hand in |
| 2 | **A.1** — the same Pipeline, one step different, and a first picture of the line | load, fit, the eight coefficients, and where the line already goes wrong · *(the three-step table is a 📖 refresher from session 1)* |
| 3 | **A.2** — a new baseline, **and a picture of what R² divides** | **why its R² comes out negative**, which every class asks |
| 4 | **A.3** — R², and one number you can say out loud | **the metric from Lecture 4**, with MAE as the plain-English one |
| 5 | **B.1** — the residual plot | a cloud means random error; a shape means the model is wrong in a pattern |
| 6 | **B.2** — the target that hit a ceiling | **965 districts stuck at 5.00001**, and the residual plot shows it instantly |
| 7 | **B.3** — outliers the way Lecture 4 finds them | standardised residuals beyond ±3 flag **70 rows**; the worst is a real 36-person district |
| 8 | **Part 2 — picking your topic** | you skim and claim, not read out loud |

**The 📖 cell closing A.3, B.4 and the appendix at the bottom (A.4) are yours to read** — about 18 minutes if you sit down with them. **A.4 is two pictures**: why this data wants the median, and why MAE and RMSE disagree about it.

> **Regression only this session.** Every one of the fifteen topics predicts a number.

---
---
# 🎬 Part 1 — The Demo · blocks A and B

**Watch, do not type along.** Each block header says which subsections are walked live (🎤)
and which are reference (📖).

---
# A · From a Label to a Number
🎤 **Walked live: all of A.** Fit the line, score it, get one number you can defend — and notice one
detail that block B comes back for.

In [ ]:
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
np.random.seed(SEED)
pd.set_option('display.width', 120)
plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True; plt.rcParams['grid.alpha'] = 0.3
print('ready')

### 🔵 A.1 — The same Pipeline, one step different

California housing: **20,640 districts**, and the target is the median house value in that district,
in hundreds of thousands of dollars. Load it, look at three rows, split it — exactly as in session 1.

In [ ]:
h = fetch_california_housing(as_frame=True)
X, y = h.data, h.target

print('shape       :', h.frame.shape)
print('target      : min %.4f   median %.4f   max %.4f' % (y.min(), y.median(), y.max()))
print()
print(h.frame.head(3).round(3).to_string())

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print()
print('train %d rows · test %d rows' % (len(X_train), len(X_test)))

✅ **Expected:** `(20640, 9)` · target runs from `0.1500` to `5.0000` with a median of `1.7970` ·
16,512 training rows and 4,128 test rows

**Hold on to that maximum of 5.0000.** It comes back in B.2 and it is not a coincidence.

Nothing about the structure changes from session 1. Impute, scale, model. **Only the last step is new** —
`LinearRegression`, **the model Lecture 4 (Regression Analysis) is entirely about**, in place of a classifier. The
least-squares line from the slides is exactly what `.fit()` computes here, on 16,512 rows and eight
columns instead of one column in Excel. `pipe()` is a shortcut so the rest of the demo can say
`pipe(SomeModel())` instead of retyping the same three steps.

In [ ]:
def pipe(model):
    return Pipeline([('impute', SimpleImputer(strategy='median')),
                     ('scale',  StandardScaler()),
                     ('model',  model)])

lin  = pipe(LinearRegression()).fit(X_train, y_train)
pred = lin.predict(X_test)

print('first five predictions :', pred[:5].round(3))
print('first five actual      :', y_test.values[:5].round(3))

✅ **Expected:** two rows of numbers that are close but never equal.

**That is the whole difference.** A classifier is right or wrong; a regressor is only ever
*off by some amount*, and the rest of this session is about how to measure that amount honestly.

**Can we see this model?** Not all of it — eight features put the line in nine dimensions. But two
pictures are standard: fit the line on **one** feature and it is a line you can draw, exactly Lecture 4's
Excel picture; for **all eight**, plot what the line predicted against what was true.

In [ ]:
one = pipe(LinearRegression()).fit(X_train[['MedInc']], y_train)
m, sc = one.named_steps['model'], one.named_steps['scale']
slope = m.coef_[0] / sc.scale_[0]                 # undo the scaler so the line reads in $10k of income
intercept = m.intercept_ - slope * sc.mean_[0]
print('the line on MedInc alone :  value = %.4f + %.4f * MedInc' % (intercept, slope))

# the 8-feature line, as the model sees it: every feature was scaled to mean 0 / spread 1 first,
# so each coefficient is "how much the prediction moves for one standard deviation of that column"
lin_model = lin.named_steps['model']
print()
print('the line on all 8 features:  intercept %.4f  (= mean of y_train %.4f)' % (lin_model.intercept_, y_train.mean()))
print(pd.Series(lin_model.coef_, index=X.columns, name='coef per std of feature').round(4).to_string())

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
ax[0].scatter(X_test['MedInc'], y_test, s=5, alpha=0.2, color='tab:blue', label='4,128 test districts')
xs = np.linspace(0, 15, 50)
ax[0].plot(xs, intercept + slope * xs, color='tab:red', lw=2, label='least-squares line, MedInc only')
ax[0].set_xlabel('MedInc  (median income, $10k)'); ax[0].set_ylabel('median house value ($100k)')
ax[0].set_title('One feature: a line you can draw', fontsize=10); ax[0].set_xlim(0, 15.5); ax[0].legend(fontsize=8)

ax[1].scatter(pred, y_test, s=5, alpha=0.2, color='tab:blue', label='4,128 test districts')
ax[1].plot([-1, 6], [-1, 6], color='tab:red', lw=2, label='perfect prediction  (actual = predicted)')
ax[1].set_xlabel('predicted by the 8-feature line'); ax[1].set_ylabel('actual value')
ax[1].set_title('Eight features: actual against predicted', fontsize=10)
ax[1].set_xlim(-1, 6.5); ax[1].set_ylim(-0.2, 5.4); ax[1].legend(fontsize=8, loc='upper left')
fig.tight_layout(); plt.show()

print('predictions above 5.0      : %d districts   (the line has no ceiling)' % (pred > 5).sum())
print('predictions past the right edge of the plot (> 6.5): %d, the farthest at %.1f  -- cropped so the cloud stays readable'
      % ((pred > 6.5).sum(), pred.max()))

✅ **Expected:** `value = 0.4446 + 0.4193 * MedInc`, a cloud with one red line through it on the left,
and on the right a cloud leaning along the diagonal · `39` districts predicted above 5.0 — the line has no
ceiling, even though the data does — and `12` of them past the plot's right edge, the farthest at `11.5`
(cropped on purpose; B.3 goes and finds it).

**The eight coefficients are the line.** Each one says how far the prediction moves when that column goes up
by one standard deviation (the scaler put every column on that footing). `MedInc` **+0.8544**: one
standard deviation more income, about \$85,000 more. `Latitude` **−0.8969** and `Longitude` **−0.8698**:
further north and further east — away from the coast — cheaper. `Population` **−0.0023**: the line barely
uses it. The intercept **2.0719** is exactly the training mean — with every column at its average, the
line predicts the average. *(Why the two location coefficients cannot be trusted one at a time is B.4.)*

**On the right, the red diagonal is "predicted exactly right"** — the closer the cloud hugs it, the
better the model. And look at the **flat row at 5.0** in both panels: every district there has the same
true value. Hold that too. **B.2.**

#### 📖 Read later — what the three steps are, since we build one every session

*A refresher from session 1, not walked today.*

A **`Pipeline`** is a list of steps that run in order. Everything before the last one *transforms* the
data; the last one is the *model*. You call `.fit()` once, on the whole thing.

| Step | What it does | What it is protecting you from |
|---|---|---|
| `impute` | fills in missing values — here with the **median of that column**. *Median rather than mean: the columns are skewed (`Population` mean 1426, median 1167), and a filled-in value should look like a typical row, not the tail — A.4 prints both numbers and draws the histogram* | a model that refuses to run, or silently drops rows |
| `scale` | rewrites every column to mean 0, spread 1, so **no column is "bigger" than another just because of its units** | a model that thinks population matters more than income because the numbers are larger |
| `model` | the part that actually learns | — |

**The reason it is a `Pipeline` and not three separate lines is leakage.** Each step learns its numbers
(the median to fill with, the mean and spread to scale by) **from the training data only**, and then
applies those same numbers to the test data. Do it by hand on the full table before splitting and the
test set has already leaked into your preprocessing — the scores come out better than the truth. That
was C.6 in session 1, and it is why the split came first and the `Pipeline` after it.

> **Neither transformer does anything visible on *this* dataset.** California housing has **zero
> missing values**, and a least-squares line fitted to rescaled columns is **the same line** — only the
> coefficients change units — so MAE is `0.5332` with or without the scaler. **Both steps stay:** your
> own topic will have missing values, and a model that measures distance — session 3's k-Means — cannot
> work without `scale`.

### 🔵 A.2 — The baseline changes too — and its R² goes negative

Session 1's baseline guessed the most common class. **There is no most common house price**, so the
dumbest possible answer becomes: *predict the training median, every time, for everyone.*

`DummyRegressor` does exactly that. Session 1 used `DummyClassifier`; this is the same idea with
the only strategy that makes sense for a number.

In [ ]:
dumb = DummyRegressor(strategy='median').fit(X_train, y_train)
base_pred = dumb.predict(X_test)

print('baseline = always predict the training median (%.4f)' % y_train.median())
print('  MAE  %.4f' % mean_absolute_error(y_test, base_pred))
print('  R2   %.4f' % r2_score(y_test, base_pred))

✅ **Expected:** median `1.7985` · MAE `0.8740` · **R² `-0.0502`**

**Every year somebody asks why R² is negative.** One picture answers it — and shows what R² actually
is: a ratio of two sets of errors, how wrong *your model* is over how wrong *always guessing the mean*
would be.

In [ ]:
rng  = np.random.RandomState(0)
idx  = np.sort(rng.choice(len(y_test), 35, replace=False))
act  = y_test.values[idx]
mref = y_test.mean()

ss_tot = ((y_test - mref) ** 2).sum()     # how wrong the mean is
ss_res = ((y_test - pred) ** 2).sum()     # how wrong our model is

fig, ax = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
x = np.arange(len(idx))
ax[0].vlines(x, pred[idx], act, color='tab:red', lw=2, alpha=0.55)
ax[0].plot(x, pred[idx], 's', ms=4, color='k', label='model prediction')
ax[0].axhline(mref, color='k', lw=1.6, label='mean of the test set  (R2 = 0)')
ax[0].axhline(y_train.median(), color='tab:orange', ls='--', lw=1.4, label='our baseline: training median')
ax[0].set_title('Red: how wrong the model is', fontsize=10)
ax[0].set_ylabel('median house value ($100k)')
ax[1].vlines(x, mref, act, color='tab:grey', lw=2, alpha=0.55)
ax[1].axhline(mref, color='k', lw=1.6, label='mean of the test set  (R2 = 0)')
ax[1].axhline(y_train.median(), color='tab:orange', ls='--', lw=1.4,
              label='our baseline: training median')
ax[1].set_title('Grey: how wrong always guessing the mean is', fontsize=10)
for a in ax:
    a.plot(x, act, 'o', ms=5, color='tab:blue', label='actual value')
    a.set_xlabel('35 test districts, drawn at random')
    a.legend(fontsize=8, loc='upper right')
fig.suptitle('R2  =  1 - (sum of red squared) / (sum of grey squared)'
             '  =  1 - %.0f / %.0f  =  %.4f' % (ss_res, ss_tot, 1 - ss_res / ss_tot), fontsize=11)
fig.tight_layout(); plt.show()

print('sum of grey squared  (how wrong the mean is)  : %.0f' % ss_tot)
print('sum of red squared   (how wrong the model is) : %.0f' % ss_res)
print('R2 = 1 - %.0f / %.0f = %.4f' % (ss_res, ss_tot, 1 - ss_res / ss_tot))

✅ **Expected:** red lines on the left, grey lines on the right that are visibly longer,
and a title reading `1 - 2295 / 5409 = 0.5758`.

**Read it as one sentence: R² is the share of the squared error that your model removed** — all of it
gives 1, none of it gives 0, and making things worse than the mean goes negative.

**So is R² near 1 a good model?** Closer to 1 means less error left, *on this test set*. Two cautions.
An R² of 0.99 on a first attempt is usually a leak, not a triumph (session 1 C.6: a column that already
knows the answer). And our 0.5758 looks respectable, yet block B finds a bend, a ceiling and an outlier
it never noticed. **R² says how much error is left, not where it is.**

**The two horizontal lines on the right panel are the whole reason the baseline scored below zero.** The
solid black line is what R² = 0 means: the mean of the *test* set, which you never get to see. The dashed
orange line is what our baseline actually predicts: the median of the *training* set. Close, but not the
same line — and the baseline is measured against the black one, so it lands just below zero.
**A baseline R² near zero, on either side, is correct.** What would be alarming is a baseline R² of 0.4.

**The same black and orange lines are drawn on the left panel too**, so you can see the model's
predictions (the black squares) moving off them — that is what the slopes bought.

*(Why the baseline predicts the median rather than the mean, and why that alone accounts for the
−0.05: A.4, two pictures.)*

### 🔵 A.3 — R², and one number you can say out loud

> **R² is the metric from Lecture 4** — the same one in Excel's regression output. The second column below,
> MAE, is not in any deck; it is here only because it is the one number a non-specialist understands.

The two-panel R² figure in A.2 scored one model. Now the table: the baseline, the line, and — for comparison only — a tree
like session 1's. **MAE is the second column: the average size of a miss, in the target's own units.**

In [ ]:
def report(name, y_pred):
    return {'model': name,
            'MAE': mean_absolute_error(y_test, y_pred),
            'R2':  r2_score(y_test, y_pred)}

rows = [report('Baseline (train median)', pipe(DummyRegressor(strategy='median')).fit(X_train, y_train).predict(X_test)),
        report('Linear Regression',       pipe(LinearRegression()).fit(X_train, y_train).predict(X_test)),
        report('Decision Tree (depth 8)', pipe(DecisionTreeRegressor(max_depth=8, random_state=SEED)).fit(X_train, y_train).predict(X_test))]

print(pd.DataFrame(rows).round(4).to_string(index=False))

✅ **Expected**

| model | MAE | R² |
|---|--:|--:|
| Baseline (train median) | 0.8740 | −0.0502 |
| Linear Regression | 0.5332 | 0.5758 |
| Decision Tree (depth 8) | 0.4482 | 0.6779 |

**One row at a time.**

- **Baseline** — guesses the training median, 1.7985 (printed in A.2), for every district. Off by **0.8740** on average. R² is about 0,
  because "guess one number for everyone" is what R² = 0 means.
- **Line** — off by **0.5332** on average. That is about \$53,000 per district. Its R² of **0.5758**
  is the number from the R² figure in A.2, `1 − 2295/5409`: of the error the do-nothing guess makes, the line
  got rid of 58%.
- **Tree** — off by **0.4482**, R² **0.6779**. Better than the line. *Why* is block B.

**The two columns say the same thing in two languages.**

- **MAE** is in dollars. *"Typically off by \$53,000."* Anyone understands it.
- **R²** is a share, no units. *"Removed 58% of the do-nothing error."* Use it to compare across
  datasets.

**Never say a score on its own — say it next to the baseline.** "MAE 0.5332" means nothing. "MAE went
from 0.8740 to 0.5332, **39.0% less error**" — `(0.8740 − 0.5332) / 0.8740` — is a sentence that
survives a question.

*(RMSE, the third metric you will meet in the wild, is the 📖 cell just below — not walked.)*

#### 📖 Read later — the third metric, and how to choose between them

*Not walked. Read on your own — RMSE turns up in every paper and every Kaggle leaderboard,
so you will meet it whether or not it is in a lecture.*

**All three metrics are built from one quantity — the error on a single row** — and differ only in
what they do with it:

```
    error on one row      e = actual - predicted

    MAE  = mean( |e| )                                  average size of a miss, in the target's units
    RMSE = sqrt( mean( e^2 ) )                          the same, after squaring: big misses count more
    R2   = 1 - sum( e^2 ) / sum( (actual - mean)^2 )    share of the squared error the model removed
```

**MAE and RMSE differ by one step: the squaring.** Squaring is what makes a single large miss outweigh
several small ones, which is why RMSE is never below MAE. On our line, MAE is `0.5332` (the A.3 table) and RMSE is
`0.7456` (`mean_squared_error(y_test, pred) ** 0.5` — run it); the gap between them says a few large misses are doing most of the damage — B.2 will show
you exactly which rows.

**Which to use is a decision about consequences, not about which number looks better.** Ask one question:

> **Is one large error worse than several small ones that add up the same?**

| Your answer | The metric | Because |
|---|---|---|
| **Yes** — one district valued \$200k out is worse than four out by \$50k | **RMSE** | squaring makes the large miss dominate |
| **No** — the total is what matters | **MAE** | every error counts once, in real units |
| *"I need to compare against a different dataset"* | **R²** | unitless, but say what the baseline was |

**For house valuation a single large error usually is worse** — one wildly mispriced district is a real
problem, four slightly-off ones are noise. So RMSE is defensible here. **We report MAE alongside it**
because it is the number you can explain.

> **Decide before you train.** A metric chosen after you have seen the scores is a metric chosen to
> flatter them, and that is the difference the marking looks for.

---
# B · Reading the Errors
🎤 **Walked live:** B.1 · B.2 · B.3 &nbsp;·&nbsp; 📖 **read on your own:** B.4

**Block A ended on R² `0.5758` and a tree that beat the line, and neither number says why.** This block
is Lecture 4's residual analysis (pages 15–21) done on real data: check a fitted line by looking at what it
got wrong. One plot turns up **three things R² could not see** — a bend (B.1), a ceiling (B.2) and an
outlier (B.3).

### 🔵 B.1 — The residual plot: a cloud, or a shape

> **You have met this one** — Lecture 4 spends its middle third on residual analysis, and page 17 says what
> to look for: *"Linearity — examine scatter diagram (should appear linear)."* This is that diagram.

A **residual** is one number: actual minus predicted. **No absolute value, on purpose** — the sign says
which way the line is wrong: positive, the true price is higher than the line said (it guessed low);
negative, it guessed high. A.3 took `|e|` and `e²` to add errors up into one score; here we keep the sign
because *which way* is the whole point. Plot residuals against what the model predicted.

- **A shapeless cloud centred on zero** means the model's errors are random. That is as good as it gets.
- **Any shape at all** — a slope, a curve, a wall — means the model is wrong **in a pattern**, and a
  pattern is something you can go and fix.

In [ ]:
resid = y_test - pred
bins  = pd.cut(pred, [-20, 0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4, 20])   # $50k bands of predicted value; -20 and 20 just catch the two tails
grp   = pd.DataFrame({'pred': pred, 'resid': resid.values}).groupby(bins, observed=True)
band  = grp.mean()                      # each red dot: x = mean prediction in the band, y = mean residual
band['districts'] = grp.size()

plt.figure(figsize=(7, 4.2))
plt.scatter(pred, resid, s=6, alpha=0.2, color='tab:blue')
plt.plot(band['pred'], band['resid'], 'o-', color='tab:red', lw=2, ms=6, label='mean residual per band')
plt.axhline(0, color='k', lw=1); plt.xlim(-1.2, 7.6); plt.ylim(-4.5, 4.5)
plt.xlabel('predicted value'); plt.ylabel('residual  (actual - predicted)')
plt.title('The red line should be flat. It bends: the relationship is not a line.')
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

print('mean residual overall : %+.4f' % resid.mean())
print('predictions below zero: %d districts   (a house worth less than nothing)' % (pred < 0).sum())
print()
print('the red line, one row per band  (x = mean predicted value in that band, y = mean residual):')
print(band.rename(columns={'pred': 'x = mean predicted', 'resid': 'y = mean residual'}).round(3).to_string())

✅ **Expected:** mean residual about `+0.0035` — near zero, as it should be — `15` districts predicted
below zero, and a plot that is **anything but a shapeless cloud.** *(The red line is the mean residual in
each \$50k band of predictions — nine bands from "below 0.5" to "above 4". The table under the plot gives
each dot's coordinates: x is the mean prediction inside the band, not the band's midpoint, which is why the
last dot sits near 5 and not at 12; the bands hold 96 to 1,010 districts each, so the bends are real.)* Four things to see, in the order we take them:

1. **The red line bends.** If the line had the right shape it would sit flat on zero. It does not —
   **that is why the tree beat the line in A.3**, and it is Lecture 4's linearity check (page 17) failing in front of you.
2. **Fifteen predictions below zero.** A house worth less than nothing. A straight line has no floor.
3. **A perfectly straight diagonal edge.** Every point on it has the same true value. **B.2.**
4. **One point off the chart** — predicted 11.5, true value 1.6. **B.3.**

### 💥 B.2 — The target hit a ceiling, and the residuals show it

> **🆕 New.** *Censored*, *truncated* and *capped* appear in no lecture deck. It is here because it is
> common in real data, **invisible to every metric**, and visible in the B.1 residual plot you just made.

Look again at A.1: the maximum target value was **5.0000**. Not 4.97, not 5.13. Exactly the maximum.

**That is the signature of a censored variable.** This data is the 1990 US Census, one row per block group.
The census form asked house value in brackets, and the top bracket was *"\$500,000 or more"* — so every
block group whose median fell in it was recorded as 500,001, which is `5.00001` here. Not a price: *"at
least this much."* Count them — and check two other columns the same way.

In [ ]:
cap = y.max()
n_cap = (y == cap).sum()
print('the maximum value is exactly %r' % cap)
print('rows sitting on it          : %d of %d  (%.2f%%)' % (n_cap, len(y), n_cap / len(y) * 100))
print()
print(y.value_counts().sort_index(ascending=False).head(4).to_string())

print()
print('two more columns are capped the same way:')
for col, note in [('MedInc', 'top bracket "$150,000 or more"'), ('HouseAge', 'top bracket "built 1939 or earlier"')]:
    top = X[col].max()
    print('   %-9s max %-8g  rows on it %5d   %s' % (col, top, (X[col] == top).sum(), note))

capped = y_test == cap
print()
print('mean residual on capped rows : %+.4f' % resid[capped].mean())
print('mean residual on all the rest : %+.4f' % resid[~capped].mean())

plt.figure(figsize=(7, 4.2))
plt.scatter(pred[~capped.values], resid[~capped], s=6,  alpha=0.25, color='0.6', label='everything else')
plt.scatter(pred[capped.values],  resid[capped],  s=10, alpha=0.8, color='tab:red',  label='actual value = 5.00001 (capped)')
plt.axhline(0, color='k', lw=1); plt.xlim(-1.2, 7.6); plt.ylim(-4.5, 4.5)
plt.xlabel('predicted value'); plt.ylabel('residual')
plt.title('The diagonal edge is every capped district, and nothing else')
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

✅ **Expected:** the maximum is `5.00001` · **965 districts (4.68%) sit exactly on it** · `MedInc` is capped
too, at `15.0001` (49 districts), and `HouseAge` at `52` (1,273 districts) · and the residuals split:

| rows | mean residual |
|---|--:|
| on the cap | **+1.1282** |
| everything else | −0.0475 |

**Every capped district is worth more than the file says** — the line under-predicts them by more than
a whole unit, and everything else by almost nothing. In the plot this cell just drew, the diagonal edge
from B.1 is now **entirely red**: it is the capped districts, and nothing else.

**No model can fix this.** A district worth \$800k was written down as \$500k before you ever saw the
file. *(Checking every column's maximum for a pile of rows sitting exactly on it is one line, and it finds
this every time — `MedInc` and `HouseAge` have the same pile.)* The honest move is to **say so in your report**, and possibly to set the capped rows aside and say why.

> R² said the line explains 58% of the variation. The residual plot said *4.7% of your data has had
> its answer erased*. **That is what a residual plot is for.**

#### 🎨 Read later — before this plot goes on a slide

Session 1's rule was *the title is the conclusion*. This session adds four for the residual plot — and
the B.2 plot above already follows all four:

| Do | Why |
|---|---|
| **Equal limits above and below zero** | an unequal axis makes symmetric errors look one-sided |
| **A visible zero line** | it is the only reference the reader has |
| **Label the points that escape** | *"the capped districts"* beats an unexplained smear |
| **Fade the bulk, colour what you are pointing at** | grey for the cloud, red for the story |

A chart titled "Residuals" is a cell output. This one is a slide.

### 🔵 B.3 — Outliers, the way Lecture 4 finds them: standardised residuals

> **This is straight from the lecture.** Lecture 4 page 15: *"Standard residual = residual / standard
> deviation. Rule of thumb: standard residuals outside of ±2 or ±3 are potential outliers."* Page 16
> then finds a home with a standardised residual over 4 and asks what is unusual about it. Same here.

Divide every residual by the standard deviation of all residuals. Now a residual of 2 means *two
standard deviations further off than typical* — the same yardstick for every row, whatever its units.

In [ ]:
z = resid / resid.std()

for k in (2, 3):
    print('|z| > %d : %4d districts  (%.2f%%)' % (k, (z.abs() > k).sum(), (z.abs() > k).mean() * 100))

worst = z.abs().sort_values(ascending=False).head(3).index
print()
print('the three most extreme, with the columns that explain them:')
print(pd.DataFrame({'z':          z[worst].round(2),
                    'actual':     y_test[worst].round(3),
                    'predicted':  pd.Series(pred, index=y_test.index)[worst].round(3),
                    'AveRooms':   X_test.loc[worst, 'AveRooms'].round(2),
                    'Population': X_test.loc[worst, 'Population'].astype(int)}).to_string())

ok, look, out = (z.abs() <= 2).values, ((z.abs() > 2) & (z.abs() <= 3)).values, (z.abs() > 3).values
plt.figure(figsize=(7, 4.2))
plt.scatter(pred[ok],   z[ok],   s=6,  alpha=0.2, color='tab:blue',   label='|z| <= 2')
plt.scatter(pred[look], z[look], s=12,            color='tab:orange', label='2 < |z| <= 3   worth a look')
plt.scatter(pred[out],  z[out],  s=14,            color='tab:red',    label='|z| > 3   potential outlier')
for k in (-3, -2, 2, 3):
    plt.axhline(k, color='grey', ls='--', lw=1)
plt.xlabel('predicted value'); plt.ylabel('standardised residual  z')
plt.title("Lecture 4's rule of thumb, drawn: past 2 look, past 3 potential outlier")
plt.legend(fontsize=8, loc='upper right'); plt.tight_layout(); plt.show()

✅ **Expected:** `220` districts beyond ±2 (5.33%) · `70` beyond ±3 (1.70%) · and the three most extreme:

| | z | actual | predicted | AveRooms | Population |
|---|--:|--:|--:|--:|--:|
| 1979 | **−13.24** | 1.625 | 11.500 | **132.53** | **36** |
| 6688 | +5.56 | 5.000 | 0.852 | 7.68 | 142 |
| 10574 | +5.21 | 5.000 | 1.115 | 4.80 | 125 |

**Two things to see in the plot.** The red points above +3 form a straight run — **the capped districts
from B.2**, caught by the rule without being told about the cap. The one red point far below everything
is the first row of the table: **a district of 36 people averaging 132 rooms per household.** The line
saw 132 rooms and priced it at 11.5.

**Is 132 rooms a typo? No.** The dataset's own documentation says these values occur in block groups
*"with few households and many empty houses, such as vacation resorts"* — thirty-six residents, and a
great many holiday homes standing empty on census day. A rare, real value.

**The rule finds candidates. You make the decision.** A recording error (an age of 999) you fix or drop,
and say so. A rare real value you keep and explain — or fit without it and report both numbers. Drop
everything past ±3 here and you throw away 70 districts, most of them real. Lecture 4 page 16 makes
exactly this decision out loud.

### 📖 B.4 — Multicollinearity: two columns saying the same thing

Lecture 4 pages 30–35 introduce this through the correlation matrix, then show what it does to the coefficients. Both halves, on our data.

In [ ]:
corr = X.corr()
pairs = [(a, b, corr.loc[a, b]) for i, a in enumerate(corr.columns) for b in corr.columns[i + 1:]]
pairs.sort(key=lambda t: -abs(t[2]))

print('most correlated feature pairs:')
for a, b, v in pairs[:3]:
    print('  %-12s %-12s %+.4f' % (a, b, v))

print()
print('correlation of each feature with the target:')
print(X.corrwith(y).sort_values(key=abs, ascending=False).round(4).to_string())

plt.figure(figsize=(7, 5.5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, square=True, cbar=False)
plt.title('Correlation between features: the two dark cells are one piece of information')
plt.tight_layout(); plt.show()

# the part a tree never shows you: what a correlated pair does to the coefficients
coef = pd.Series(lin.named_steps['model'].coef_, index=X.columns)
without = [c for c in X.columns if c != 'Longitude']
lin_wo  = pipe(LinearRegression()).fit(X_train[without], y_train)
coef_wo = pd.Series(lin_wo.named_steps['model'].coef_, index=without)
print()
print('coefficient of Latitude  with Longitude in the model : %+.4f' % coef['Latitude'])
print('coefficient of Latitude  with Longitude removed      : %+.4f' % coef_wo['Latitude'])
print('R2 with Longitude %.4f   without %.4f' % (r2_score(y_test, pred), r2_score(y_test, lin_wo.predict(X_test[without]))))

✅ **Expected:** `Latitude` and `Longitude` at **−0.9247**, `AveRooms` and `AveBedrms` at **+0.8476**

**Latitude and longitude are almost perfectly anti-correlated here** because California runs on a
diagonal — go north and you also go west. The two columns are largely one piece of information.

**What it costs you — and now you can see it:** Latitude's coefficient is **−0.8969** with Longitude
in the model and **−0.0639** without it, a fourteen-fold swing from removing one correlated partner,
while R² barely moves (`0.5758` → `0.5139`). The two columns share the job, so the model cannot say
which one is doing it. **Any statement of the form "latitude matters more than longitude" is not
supported by this data** — which is Lecture 4's point on page 30, made with our numbers.

**And notice `MedInc` at 0.6881 against the target**, four times anything else. Whatever your model
does, median income is most of what it has to work with.

---
# 📖 Appendix — A.4, for reading after class
**Not walked.** Two pictures that answer one question this session leaves open: *why median?*

### 📖 A.4 — Why median: two pictures

The pipeline in A.1 fills missing values with the **median**, and the baseline in A.2 predicts the
**median**. Both choices come from one fact about this data, and both have a cost that A.2's negative
R² already showed you. Two pictures explain it.

#### Picture 1 — the data is skewed, and the mean sits in the tail

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, (series, name) in zip(ax, [(X_train['Population'], 'Population'),
                                  (y_train, 'target (median house value)')]):
    hi = series.quantile(0.99)                       # clip the top 1% so the picture is readable
    a.hist(series.clip(upper=hi), bins=60, color='tab:grey', alpha=0.7)
    a.axvline(series.mean(),   color='tab:red',  lw=2,          label='mean  %.2f'   % series.mean())
    a.axvline(series.median(), color='tab:blue', lw=2, ls='--', label='median  %.2f' % series.median())
    a.set_title('%s  -  the mean is pulled into the tail' % name, fontsize=10)
    a.set_xlabel(name); a.legend(fontsize=8)
ax[0].set_ylabel('districts'); fig.tight_layout(); plt.show()

below = (X_train['Population'] < X_train['Population'].mean()).mean() * 100
print('Population: mean %.2f   median %.2f   -> %.1f%% of districts are BELOW the mean'
      % (X_train['Population'].mean(), X_train['Population'].median(), below))

✅ **Expected:** both histograms lean left with a long right tail · the red mean line sits to the right of
the dashed median in each · and **63.5% of districts have a population below the "average"** —
mean `1426.45`, median `1167.00`.

**Skew makes the mean untypical.** Fill a missing `Population` with 1426 and you have invented a
district larger than two-thirds of the real ones; 1167 is a district you could actually find. That is
why `impute` uses the median. *(The spike at 5.0 on the right panel is B.2's capped target.)*

#### Picture 2 — why MAE likes the median and RMSE likes the mean

The two metrics charge a miss differently. Absolute error charges a miss of 3 as 3; squared error
charges it as 9:

In [ ]:
e = np.linspace(-3, 3, 601)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(e, np.abs(e), color='tab:blue', lw=2, label='absolute error  |e|')
ax.plot(e, e ** 2,    color='tab:red',  lw=2, label='squared error  e2')
for pt in (1, 2, 3):
    ax.plot([pt, pt], [pt, pt ** 2], color='k', lw=1, ls=':')
    ax.annotate('%d vs %d' % (pt, pt ** 2), (pt, pt ** 2), textcoords='offset points', xytext=(6, -2), fontsize=9)
ax.set_xlabel('error on one row,  e = actual - predicted'); ax.set_ylabel('how much that row costs')
ax.set_title('Squared error charges a miss by its square, so far rows dominate', fontsize=10)
ax.legend(fontsize=9); fig.tight_layout(); plt.show()

✅ **Expected:** a V and a parabola that cross at `e = 1` · past that the parabola runs away.

So far-away rows barely register with MAE and dominate RMSE. Now guess one constant `c` for every
district and slide it from 1 to 3 — watch where each metric is happiest:

In [ ]:
cs   = np.linspace(1.0, 3.0, 401)
mae  = np.array([np.abs(y_test - c).mean() for c in cs])
rmse = np.array([np.sqrt(((y_test - c) ** 2).mean()) for c in cs])

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(cs, mae, color='tab:blue')
ax[0].axvline(y_test.median(), color='k', ls='--', lw=1.2, label='median of y_test')
ax[0].set_title('MAE of "always predict c"  -  lowest at the median', fontsize=10)
ax[1].plot(cs, rmse, color='tab:red')
ax[1].axvline(y_test.mean(), color='k', ls='--', lw=1.2, label='mean of y_test')
ax[1].set_title('RMSE of "always predict c"  -  lowest at the mean', fontsize=10)
for a in ax:
    a.set_xlabel('the constant c you predict for every row'); a.legend(fontsize=8)
ax[0].set_ylabel('error'); fig.tight_layout(); plt.show()

print('MAE  is lowest at c = %.4f   median of y_test = %.4f' % (cs[mae.argmin()],  y_test.median()))
print('RMSE is lowest at c = %.4f   mean   of y_test = %.4f' % (cs[rmse.argmin()], y_test.mean()))

✅ **Expected:** the MAE curve bottoms at `1.7850` — the nearest grid point to the median, `1.7865` — and
the RMSE curve at `2.0550`, which is the mean exactly.

**MAE is lowest at the median, RMSE at the mean.** Absolute error counts every row once, so the best
constant is the one with half the rows on each side — the median. Squared error charges by distance
squared, so far rows pull the best constant toward them — the mean, dragged into the tail.

**And that is A.2's negative R².** Our baseline predicts the median (the best constant for MAE) but R² is
a squared-error score, whose zero point is the mean. Judged by the wrong metric's favourite, the median
lands just below zero. *Pick the metric first, then the baseline that suits it.*

---
---
# 📋 Part 2 — Pick Your Topic
### Assignment 2 · one of these 15 · this takes ~8 minutes

**Groups of three or four. One topic per group, first come first served, no two groups on the same one.**

**Every topic here predicts a number.** Take yours and go the whole way:

```
your data -> EDA -> preprocessing (Pipeline) -> baseline() -> model -> a metric you can justify -> limitations
```

📄 **[How it is marked, and what to hand in](https://classes.incortx.com/DataAnalytics/session-00/)** — the short version: you need **a baseline beside
every number**, a **`Pipeline` with no leakage**, and **a metric chosen from what an error costs**.
A model that loses to its own baseline still scores full marks if you can explain why.

**The 15 topics.** 🟢 straightforward · 🟡 has something awkward in it

The **trap** column is not a general warning. It is the specific thing that will bite you in that
dataset, and it is where the questions will come from.

| # | Topic | Data | The trap | |
|:--:|---|---|---|:--:|
| **1** | What is this district of houses worth? | `fetch_california_housing()` | **The target is capped at 5.0** -- 965 of 20,640 districts (4.7%) sit exactly on the ceiling, and no amount of tuning will ever get those right | 🟢 |
| **2** | What should this diamond cost? | `sns.load_dataset('diamonds')` | The `cut`/`color`/`clarity` grades are ordinal and **sort into the wrong order alphabetically** -- `IF`, the best clarity there is, lands second | 🟢 |
| **3** | How much forest will this fire burn? | `archive.ics.uci.edu/static/public/162/forest+fires.zip` | **247 of 517 rows have target = 0.00.** Predicting zero every time looks respectable on average -- you have to decide what you are actually predicting | 🟡 |
| **4** | How many bikes will be hired this hour? | `archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip` (use `hour.csv`) | `casual` + `registered` equals the target **exactly, on all 17,379 rows.** Leave either one in and you score almost perfectly without predicting anything | 🟢 |
| **5** | How much fuel does this car use? | `sns.load_dataset('mpg')` | `name` is unique on 305 of 398 rows -- feeding it in lets the model memorise cars one at a time. And `horsepower` has 6 NaNs you have to decide about | 🟢 |
| **6** | Predict diabetes progression one year ahead | `sklearn.datasets.load_diabetes()` | The features **arrive already standardised** (every column has mean zero), so the coefficients you get cannot be read back in real units | 🟡 |
| **7** | How much load will this concrete mix take? | `archive.ics.uci.edu/static/public/165/concrete+compressive+strength.zip` | The ingredients are **compositional** -- they sum to a fixed weight, so the features are not independent. And curing age acts logarithmically, not linearly | 🟡 |
| **8** | How much heating energy does this building need? | `archive.ics.uci.edu/static/public/242/energy+efficiency.zip` | **There are two targets** (heating load and cooling load). Pick one or go multi-output -- pick the wrong one and your numbers cannot be compared with anyone else's | 🟢 |
| **9** | What is this Taipei apartment worth? | `archive.ics.uci.edu/static/public/477/real+estate+valuation+data+set.zip` | Only 414 rows, and **`transaction date` is a decimal year** (2013.250). Use it raw and the model learns the time trend instead of the location | 🟢 |
| **10** | How old is this abalone? | `archive.ics.uci.edu/static/public/1/abalone.zip` | **It looks like classification and is really ordinal** -- being wrong by 1 year and being wrong by 10 should not cost the same | 🟡 |
| **11** | How loud is this wing shape? | `archive.ics.uci.edu/static/public/291/airfoil+self+noise.zip` | Unusually clean, not a single missing value -- **which makes it the topic with no excuses.** A poor result means the model or the metric is wrong, not the data | 🟢 |
| **12** | How many bikes will Seoul hire? | `archive.ics.uci.edu/static/public/560/seoul+bike+sharing+demand.zip` | Encoded as **cp949, not utf-8** -- opening it directly fails. Fix with `pd.read_csv(f, encoding="cp949")`. Holidays and seasons also have to be turned into features yourself | 🟡 |
| **13** | What grade will this student get? (as a number) | `archive.ics.uci.edu/static/public/320/student+performance.zip` | **G1 and G2 predict G3 almost perfectly (corr 0.905)** -- keeping them gives a lovely score and tells you nothing. Decide, and explain the decision | 🟢 |
| **14** | What score will this wine get? (as regression) | `archive.ics.uci.edu/static/public/186/wine+quality.zip` | The target is an integer from 3 to 8. **Treat it as regression and you get decimals -- decide whether to round**, and know that rounding moves the score a lot | 🟡 |
| **15** | What should this house sell for? | `fetch_openml('house_prices', version=1)` (Ames, 80 columns) | **80 columns, several of them more than half empty** -- you decide column by column what to keep. The heaviest preprocessing job in the bank | 🟡 |

In [ ]:
# ── Record your group's choice ─────────────────────────────────────────────
TOPIC_ID = None      # <- put your group's topic number here, then run this cell

TOPIC_TRAPS = {
     1: ('What is this district of houses worth?',
        '**The target is capped at 5.0** -- 965 of 20,640 districts (4.7%) sit exactly on the ceiling, and no amount of tuning will ever get those right'),
     2: ('What should this diamond cost?',
        'The `cut`/`color`/`clarity` grades are ordinal and **sort into the wrong order alphabetically** -- `IF`, the best clarity there is, lands second'),
     3: ('How much forest will this fire burn?',
        '**247 of 517 rows have target = 0.00.** Predicting zero every time looks respectable on average -- you have to decide what you are actually predicting'),
     4: ('How many bikes will be hired this hour?',
        '`casual` + `registered` equals the target **exactly, on all 17,379 rows.** Leave either one in and you score almost perfectly without predicting anything'),
     5: ('How much fuel does this car use?',
        '`name` is unique on 305 of 398 rows -- feeding it in lets the model memorise cars one at a time. And `horsepower` has 6 NaNs you have to decide about'),
     6: ('Predict diabetes progression one year ahead',
        'The features **arrive already standardised** (every column has mean zero), so the coefficients you get cannot be read back in real units'),
     7: ('How much load will this concrete mix take?',
        'The ingredients are **compositional** -- they sum to a fixed weight, so the features are not independent. And curing age acts logarithmically, not linearly'),
     8: ('How much heating energy does this building need?',
        "**There are two targets** (heating load and cooling load). Pick one or go multi-output -- pick the wrong one and your numbers cannot be compared with anyone else's"),
     9: ('What is this Taipei apartment worth?',
        'Only 414 rows, and **`transaction date` is a decimal year** (2013.250). Use it raw and the model learns the time trend instead of the location'),
    10: ('How old is this abalone?',
        '**It looks like classification and is really ordinal** -- being wrong by 1 year and being wrong by 10 should not cost the same'),
    11: ('How loud is this wing shape?',
        'Unusually clean, not a single missing value -- **which makes it the topic with no excuses.** A poor result means the model or the metric is wrong, not the data'),
    12: ('How many bikes will Seoul hire?',
        'Encoded as **cp949, not utf-8** -- opening it directly fails. Fix with `pd.read_csv(f, encoding="cp949")`. Holidays and seasons also have to be turned into features yourself'),
    13: ('What grade will this student get? (as a number)',
        '**G1 and G2 predict G3 almost perfectly (corr 0.905)** -- keeping them gives a lovely score and tells you nothing. Decide, and explain the decision'),
    14: ('What score will this wine get? (as regression)',
        'The target is an integer from 3 to 8. **Treat it as regression and you get decimals -- decide whether to round**, and know that rounding moves the score a lot'),
    15: ('What should this house sell for?',
        '**80 columns, several of them more than half empty** -- you decide column by column what to keep. The heaviest preprocessing job in the bank'),
}

if TOPIC_ID in TOPIC_TRAPS:
    title, trap = TOPIC_TRAPS[TOPIC_ID]
    print('Topic %d: %s' % (TOPIC_ID, title))
    print('Watch out for : %s' % trap)
else:
    print('Set TOPIC_ID to your group number (1-15) and run this cell again.')

✅ **Expected:** your topic and its trap printed back at you. Write the trap somewhere you will see it
again — it is the first thing to check when your results look strange.

---
---
# 🟠 Part 3 — Your Hour · 60 Minutes, Your Own Data

**Everything above was the demo.** From here it is your group's work, and it is what gets marked.

This section does not depend on a single cell above it. Run it from the top of this section and it works.

### The steps, and the clock

| Minutes | Step | What has to exist when you are done |
|:--:|---|---|
| — | **0 · Data already loaded** | you did this before class. `shape` · `head()` · one sentence on what a row is |
| 0–12 | **1 · EDA → one insight** | a chart, and a sentence stating what you *found* |
| 12–27 | **2 · Prepare the data** | what was wrong, what you did, **and why that choice** — inside a `Pipeline` |
| 27–37 | **3 · Metric + baseline** | the metric **with a reason**, and `baseline()` measured the same way |
| 37–50 | **4 · Today's technique** *(if you get there)* | a score, printed next to the baseline |
| 50–60 | **5 · Write up + get ready** | one sentence, one limitation, notebook scrolled to where you start |

**Steps 1, 2 and 3 are what you present and what is marked. Step 4 is a bonus.**

**Because loading happened before class, this hour is less rushed than session 1's.** Use the slack on
step 1 — a residual plot you actually read beats a model you cannot explain.

In [ ]:
# ── SUBMISSION HEADER — fill this in first ─────────────────────────────────
GROUP     = ''            # your group letter: 'A' .. 'J'
MEMBERS   = ['', '', '']  # everyone in the group - keep this order all term
TOPIC_ID  = None          # the topic number your group claimed

# EVERY member speaks in the video. Every assignment, no exceptions.
IN_CLASS  = ['', '']      # the TWO representing the group in the room

# ── check ───────────────────────────────────────────────────────────────────
_all   = [m.strip() for m in MEMBERS  if m.strip()]
_room  = [m.strip() for m in IN_CLASS if m.strip()]

print(f'Group {GROUP or "?"} | topic {TOPIC_ID} | {len(_all)} members')
print(f'  in the room  : {", ".join(_room) or "-- nobody --"}')
print(f'  in the video : everyone - {", ".join(_all) or "-- nobody --"}')

unknown = [m for m in _room if m not in _all]
if not GROUP or not _all or TOPIC_ID is None:
    print('\n[ ] header not filled in yet')
elif unknown:
    print(f'\n[!] not found in MEMBERS: {", ".join(unknown)} - check the spelling')
elif len(_room) != 2:
    print(f'\n[!] {len(_room)} named for the room, should be exactly 2')
else:
    print('\n[ok] two representatives named, and every member speaks in the video')

### 🟠 Setup for this section

Its own imports and its own helpers, so this half runs whatever happened above.

In [ ]:
import warnings; warnings.filterwarnings('ignore')

import io as _io, zipfile, urllib.request      # several topics are zips this session
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing, fetch_openml, load_diabetes
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
np.random.seed(SEED)
pd.set_option('display.width', 120); pd.set_option('display.max_columns', 30)
plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True; plt.rcParams['grid.alpha'] = 0.3
print('ready')

**Two helpers come with the notebook: `eda()` and `baseline()`.** Run the cell below once.

**Neither does the thinking.** `eda()` prints the six views you should always look at; `baseline()`
prints the number your model has to beat. **Reading them is the marked part.**

In [ ]:
def eda(df, target=None, n=6):
    """Print the six things you should always look at first. Reading them is your job.

        eda(df)                    # no target column yet
        eda(df, target='outcome')  # adds class balance + correlation with the target
    """
    import pandas as _pd
    line = '\u2500' * 62

    print(line); print(f'1. SHAPE      {df.shape[0]:,} rows x {df.shape[1]} columns')
    print(line); print('2. ONE ROW    what does a single row actually represent?')
    print(df.head(3).to_string())

    print(line); print('3. TYPES      a number stored as text will not go into a model')
    info = _pd.DataFrame({'dtype': df.dtypes.astype(str),
                          'non_null': df.notna().sum(),
                          'distinct': df.nunique()})
    print(info.to_string())

    print(line); print('4. MISSING    how much, and in which columns')
    miss = df.isna().sum()
    miss = miss[miss > 0].sort_values(ascending=False)
    if len(miss) == 0:
        print('   isna() finds none  --  but a 0 or a -200 can be a missing value in disguise.')
        print('   Check step 5 for impossible values before you believe this.')
    else:
        print(_pd.DataFrame({'missing': miss, 'percent': (miss / len(df) * 100).round(1)}).to_string())

    print(line); print('5. RANGES     look for a min or max that cannot be real')
    num = df.select_dtypes('number')
    print(num.describe().T[['min', '25%', '50%', '75%', 'max']].to_string() if len(num.columns)
          else '   no numeric columns')

    if target is not None and target in df.columns:
        print(line); print(f'6. TARGET     {target!r}')
        y = df[target]
        if y.dtype.kind == 'f' or y.nunique() > 20:
            print(y.describe().to_string())
            corr = num.corr(numeric_only=True)[target].drop(target).sort_values(key=abs, ascending=False)
            print(f'\n   strongest correlations with {target}:')
            print(corr.head(n).round(4).to_string())
        else:
            print(y.value_counts(normalize=True).round(4).to_string())
            print(f'\n   most common class is {y.value_counts(normalize=True).max():.1%} of rows')
    print(line)
    print('Now write ONE sentence about something you did not know 60 seconds ago.')

def baseline(y_train, y_test, kind='auto'):
    """Print the score of the dumbest possible model for this target.

    You do not have to write a baseline. You do have to read it and say what it
    means -- that is the part that carries marks.

        baseline(y_tr, y_te)            # works out classification vs regression
        baseline(y_tr, y_te, 'clf')     # force it
        baseline(y_tr, y_te, 'reg')
    """
    import numpy as _np, pandas as _pd
    from sklearn.dummy import DummyClassifier, DummyRegressor
    from sklearn.metrics import (accuracy_score, f1_score, mean_absolute_error,
                                 mean_squared_error, r2_score)

    ytr, yte = _pd.Series(y_train), _pd.Series(y_test)
    if kind == 'auto':
        numeric = _pd.api.types.is_numeric_dtype(ytr)
        kind = 'reg' if (numeric and (ytr.dtype.kind == 'f' or ytr.nunique() > 20)) else 'clf'
        looks = 'a number -> regression' if kind == 'reg' else 'a label -> classification'
        print(f'[auto] your target looks like {looks}'
              f'  ({ytr.nunique()} distinct values, dtype {ytr.dtype})')
        print("       wrong guess? pass kind='clf' or kind='reg'")

    Xtr = _np.zeros((len(ytr), 1))          # a baseline ignores the features on purpose
    Xte = _np.zeros((len(yte), 1))

    if kind == 'clf':
        m = DummyClassifier(strategy='most_frequent').fit(Xtr, ytr)
        p = m.predict(Xte)
        top = m.classes_[0]
        top = top.item() if hasattr(top, 'item') else top
        acc = accuracy_score(yte, p)
        f1 = f1_score(yte, p, average='binary' if yte.nunique() == 2 else 'macro',
                      zero_division=0)
        print(f'baseline = always predict the most common class ({top!r})')
        print(f'  accuracy {acc:.4f}')
        print(f'  F1       {f1:.4f}   <- same model. If these two disagree, accuracy is the wrong metric')
        return {'accuracy': acc, 'f1': f1}

    m = DummyRegressor(strategy='median').fit(Xtr, ytr)
    p = m.predict(Xte)
    mae = mean_absolute_error(yte, p)
    rmse = mean_squared_error(yte, p) ** 0.5
    r2 = r2_score(yte, p)
    print(f'baseline = always predict the training median ({_np.median(ytr):.4f})')
    print(f'  MAE  {mae:.4f}')
    print(f'  RMSE {rmse:.4f}')
    print(f'  R2   {r2:.4f}   <- a baseline R2 at or just below zero is correct, not a bug')
    return {'mae': mae, 'rmse': rmse, 'r2': r2}

print('eda() and baseline() ready')

### 🟠 Opening a zip, if your topic is one

**Eleven of the fifteen topics are zips this session.** You should have loaded yours before class, but
here is the shape in one place.

```python
with urllib.request.urlopen(URL) as response:
    z = zipfile.ZipFile(_io.BytesIO(response.read()))
print(z.namelist())                         # always look before you read
df = pd.read_csv(z.open('the_file.csv'), sep=';')

# a zip inside a zip
inner = zipfile.ZipFile(_io.BytesIO(z.read('inner.zip')))
df = pd.read_csv(inner.open('bank-full.csv'), sep=';')
```

**Topic 12 is encoded `cp949`, not utf-8** — `pd.read_csv(f, encoding='cp949')`.
Session 1's block B has the full four-shapes reference if you need it.

### 🟠 Step 0 — Your data, already loaded

Load it and run `eda()` straight away. It takes a fraction of a second and prints all six views, so you
begin step 1 already knowing where to look.

In [ ]:
# TODO: load your group's data into `df`, then run eda() on it

df = None
# eda(df, target='...')

✅ **What topic 5's `eda()` hands you in under a second**

- **view 3 shows `name` with 305 distinct values across 398 rows** — nearly unique. Feed that to a model
  and it memorises cars one at a time.
- **view 4 shows `horsepower` missing 6 values.** Small, but you have to decide something.
- **view 6 shows the target's spread**, and which features move with it.

**Write the one sentence:** what is one row of this data?

### 🟠 Step 1 — EDA → one insight · *0–12 min*

**The chart is not the deliverable. The sentence under it is.**

For a number-valued target, the two charts that earn their place are a **histogram of the target**
(is it skewed? does it stop dead at a ceiling, like B.2?) and a **scatter of the target against your
strongest feature** (is the relationship even a straight line?).

| Not an insight | An insight |
|---|---|
| "This is a histogram of the target." | "The target stops dead at 5.0 with 965 rows on it — those are censored and no model can recover them." |
| "mpg and weight are correlated." | "Heavier cars use more fuel, but the relationship bends — a straight line will over-predict at both ends." |

In [ ]:
# TODO: one chart that shows something about your target

**What I found:** *(one sentence — a finding, not a description of the chart)*

**What this changes about what I do next:** *(write it here)*

### 🟠 Step 2 — Prepare the data · *12–27 min*

Everything step 1 told you was wrong, you now fix — **and you write down why you fixed it that way.**

**Two hard rules, and breaking either one costs you:**

1. **Put it in a `Pipeline`, do not edit `df` in place.** A `Pipeline` learns its fill values and
   scaling from the training rows only, which is the whole point.
2. **`train_test_split` comes before any `fit`.**

> **Regression adds one decision session 1 did not have: what to do about the columns that leak.**
> If a column could only be known *after* the answer, it has to go — the same `alive` problem, wearing
> a different hat. Topic 4's `casual + registered` and topic 13's `G1`/`G2` are exactly this.

In [ ]:
# TODO: split X/y, drop what leaks, then build the Pipeline

**What I fixed, and why I chose that fix:** *(one line per decision)*

### 🟠 Step 3 — Metric + baseline · *27–37 min*

**Every topic this session predicts a number.** Report **R²** (the metric from Lecture 4) and **MAE** (the
one you can say out loud), both next to the baseline. `baseline()` prints RMSE as well — the 📖 cell
closing A.3 says when that one matters.

**The baseline code came with the notebook.** One call:

```python
base = baseline(y_train, y_test)
```

**Two rules that still belong to you:**

1. **Call it with the same `y_train` / `y_test` you gave your model.**
2. **From here on, the baseline goes next to every number you print.**

**Then read what came back** — a baseline R² at or just below zero is correct, not a bug (A.2).

In [ ]:
# TODO: split, name your metric and why, then call baseline()

✅ **Expected on topic 5:** median `22.4500` · MAE `5.9788` · RMSE `7.3652` · **R² `-0.0089`**

**Our metric is ___ because ___** *(fill this in — it is half of what requirement 1 looks for)*

### 🟠 Step 4 — Today's technique · *37–50 min* — **if you get there**

**This step is a bonus, not a requirement.** Your `prep` from step 2 is already built — drop a regressor
on the end of it and score it **next to the baseline, never alone.**

In [ ]:
# TODO: put a model on the end of `prep`, and score it against the baseline

✅ **Expected on topic 5:**

| Model | MAE | RMSE | R² |
|---|--:|--:|--:|
| Baseline (median) | 5.9788 | 7.3652 | −0.0089 |
| Linear Regression | 2.2882 | 2.8877 | **0.8449** |
| Decision Tree (depth 4) | 2.3302 | 3.3721 | 0.7885 |
| Decision Tree (depth 8) | 2.1330 | 3.2953 | 0.7980 |

**On this data the line wins on R², and it lost on California in A.3** — the data decides, not the
method, which is why you always run the comparison. **Two rows of this table are your slide 8:** the
line against a tree is a comparison of two settings. **Say which way it moved and why**, and that page
is done.

### 🟠 Step 5 — Write it up and close the file properly · *50–60 min*

Two sentences, both short. **You are not presenting today**, so the rest of this goes into making the
file usable when you reopen it at home:

1. **Write down what you would try next**, while the reason is still in your head.
2. **Say where you got stuck and what you had already ruled out.** An unfinished step costs nothing;
   an unfinished step you cannot describe costs the write-up mark.

**What we found:** *(one sentence — what does your number actually mean for the question you asked?)*

**What we do not trust:** *(one limitation — something about the data or the method)*

**Where we got stuck:** *(if a step defeated you, say which and why — this is worth writing down)*

---
## 🟠 After the hour — presenting, and handing in

**Everything about how this is presented, questioned and marked lives in one place, and it is not this
notebook:**

📄 **[How the assignment works, and what to hand in](https://classes.incortx.com/DataAnalytics/session-00/)**

That page is the only version — it covers the minutes on screen, the code questions, asking
questions while other groups present, and every requirement for the hand-in. **Read it once at the
start of term, and again before your first turn.**

The three dates you need, and nothing else:

| When | What |
|---|---|
| **Before you leave today** | this notebook, as a file — *File → Download → Download .ipynb* |
| **Two days before session 3** | the homework: notebook, slides, video, README — **as files, no links** |
| **The start of session 3** | you present, from the notebook you handed in |

---
# Session 2 Wrap-Up

### Four things to remember

1. **A number-valued answer is never right, only close.** Everything this session was about measuring
   *how close*, honestly.
2. **R² is the lecture's metric; MAE is the one you can say out loud.** Report both, next to the baseline.
3. **The residual plot sees what R² cannot.** A respectable 0.58 hid a bend the line cannot follow,
   predictions below zero, and 4.7% of the target erased before we got it (B.2).
4. **Standardised residuals find candidates, not errors.** Lecture 4's ±3 rule flagged 70 districts; the worst
   was a real place with 36 people in it.

### What this session's notebook should end up carrying
- a **metric with a reason**, and `baseline()` printed beside every score
- a **residual plot you can read out loud**
- a line saying where you got to, and what you would try next

*(Dates, file formats and everything else about handing in: see the section above.)*

### Next session — no answer key at all
Two sessions of showing the model the right answers. **Next time there are none** — you group the data
without being told what the groups are, and the hard part becomes knowing whether you found anything real.